# Garment Reconstruction GPU Backend

Pipeline: **Image → 3D Mesh + Sewing Pattern**

Models: GarmentRec (mesh) + GarmentGPT (pattern) + GarmentCode (sim) + SAM2 (segmentation)

Exposed via Cloudflare Tunnel → EC2 Proxy → Frontend

**Usage:** Run "Run All" and wait for the tunnel URL at the end.

## Cell 1: Install Dependencies

In [ ]:
# Core dependencies
!pip install -q fastapi uvicorn python-multipart pillow trimesh gdown opencv-python pyyaml packaging
!pip install -q "huggingface_hub>=0.27.0" "kagglehub>=0.3.0"
!pip install -q transformers>=4.49.0

# Install SAM2 (pulls in latest torch — we force-reinstall later)
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

# FORCE-REINSTALL torch for P100 (sm_60) compatibility
# PyTorch 2.6+ dropped sm_60; 2.5.x is the latest that supports P100.
# Can't use 2.1.0+cu121 — Python 3.12 on Kaggle has no prebuilt wheel.
# Must also reinstall torchaudio from cu121 index (base image has CUDA 13 build)
!pip install torch==2.5.1+cu121 torchvision torchaudio --force-reinstall --quiet --index-url https://download.pytorch.org/whl/cu121

# Upgrade scipy after torch reinstall (numpy 2.x compat requires scipy >= 1.14)
!pip install -q --upgrade "scipy>=1.14.1"

# Reinstall onnxruntime + rembg + SAM2 after torch force-reinstall (deps got broken)
!pip install onnxruntime openmesh 2>&1 | tail -5
!pip install "rembg[cpu]" 2>&1 | tail -5
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

# Build pytorch3d from source with P100 arch (CUDA 6.0 = sm_60)
!MAX_JOBS=4 TORCH_CUDA_ARCH_LIST="6.0" pip install -q "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git"

# Install torch-scatter etc matching torch 2.5.1+cu121
!pip install -q torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

# Additional deps: GarmentRec (pymeshlab, loguru) + GarmentGPT (llamafactory, fire, vector-quantize-pytorch)
# pymeshlab needs system OpenGL libraries
!apt-get install -y -qq libgl1 libglib2.0-0 libsm6 libxext6 libxrender-dev 2>/dev/null || true
!pip install pymeshlab loguru scikit-learn torch_geometric 2>&1 | tail -5
!pip install -q llamafactory fire vector-quantize-pytorch
!pip install -q bitsandbytes
!pip install -q nest-asyncio

# Install cloudflared binary (downloaded in Cell 2 after kernel restart)
print("Cell 1 complete. Cloudflared will be downloaded in Cell 2.")

# Re-upgrade packages SAM2/base environment downgraded
!pip install -q "pillow>=12.1"

# NumPy 2.x compat shim (onnxruntime needs _blas_supports_fpe)
import numpy as _np
if not hasattr(_np._core._multiarray_umath, '_blas_supports_fpe'):
    _np._core._multiarray_umath._blas_supports_fpe = lambda *a, **k: True

# Verify critical packages install
try:
    import fastapi, uvicorn, PIL, trimesh, cv2
    print("Core packages: OK")
except ImportError as e:
    print(f"WARNING: import failed: {e}")
try:
    import pymeshlab
    print("pymeshlab: OK")
except ImportError as e:
    print(f"WARNING: pymeshlab import failed: {e}")

print("Done. All packages installed. Continuing in same kernel...")

# Clear namespace to avoid stale references, no kernel restart needed
get_ipython().run_line_magic("reset", "-f")


## Cell 2: Download Model Weights

In [ ]:
import os
import sys
import shutil
import zipfile
import numpy as np
import pickle
import glob
import subprocess
import time
import requests
from pathlib import Path
from huggingface_hub import hf_hub_download, login

# Disable Xet Storage native protocol (cold cache hangs on first access)
# Forces HTTP fallback via presigned URLs instead of hf_xet Rust downloader
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Set HF token early
os.environ["HF_TOKEN"] = ""
login(token="")
print("Logged in to HuggingFace Hub")

WD = Path("/kaggle/working")
WEIGHTS_DIR = WD / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)


# ---- Robust Download Helper (multi-layer fallback, auth headers) ----
def robust_download(url, dest, expected_mb=None, max_retries=3, desc="", extra_headers=None):
    """Download with wget (resume+progress) -> requests streaming fallback.
    extra_headers: dict of extra HTTP headers (e.g., {"Authorization": "Bearer ..."})
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    extra_headers = extra_headers or {}
    for attempt in range(max_retries):
        try:
            if desc:
                print(f"  {desc}: attempt {attempt+1}/{max_retries}...")
            # Layer 1: wget with resume, progress, retries, auth headers
            cmd = [
                "wget", "-c", "--show-progress",
                "--dns-timeout=15", "--connect-timeout=30", "--read-timeout=120",
                "--tries=3", "--retry-connrefused",
            ]
            for k, v in extra_headers.items():
                cmd += ["--header", f"{k}: {v}"]
            cmd += ["-O", str(dest), url]
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
            if r.returncode == 0 and dest.exists() and dest.stat().st_size > 0:
                size_mb = dest.stat().st_size / (1024*1024)
                if expected_mb is None or size_mb >= expected_mb * 0.9:
                    if desc: print(f"  {desc}: {size_mb:.0f} MB")
                    return True
            # Layer 2: requests streaming (bypasses wget issues, supports auth)
            if desc: print(f"  {desc}: wget failed, trying requests...")
            sess_headers = dict(extra_headers)
            with requests.get(url, headers=sess_headers, stream=True, timeout=300) as resp:
                resp.raise_for_status()
                with open(dest, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk: f.write(chunk)
            if dest.exists() and dest.stat().st_size > 0:
                size_mb = dest.stat().st_size / (1024*1024)
                if expected_mb is None or size_mb >= expected_mb * 0.9:
                    if desc: print(f"  {desc}: {size_mb:.0f} MB")
                    return True
        except Exception as e:
            if desc: print(f"  {desc}: attempt {attempt+1} failed: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
    if desc: print(f"  *** {desc}: FAILED after {max_retries} attempts ***")
    return False


# ---- Kaggle Dataset Mirror (GCP-local, no Xet, instant) ----
KAGGLE_DATASET = "jacobthankgod/korra-garment-weights"
kaggle_path = None
try:
    import kagglehub
    kp = kagglehub.dataset_download(KAGGLE_DATASET)
    if kp and Path(kp).exists():
        kaggle_path = kp
        print(f"Kaggle Dataset '{KAGGLE_DATASET}' available: {kp}")
    else:
        print(f"Kaggle Dataset '{KAGGLE_DATASET}' not found. "
              f"Upload weights first: python scripts/upload_garment_weights_to_kaggle.py")
except Exception as e:
    print(f"Kaggle Dataset not available: {e}")


# ---- cloudflared binary (must persist after kernel restart) ----
cloudflared_bin = WD / "cloudflared"
# Must be a valid ELF binary >= 1MB (corrupt if DNS failed during download)
if not cloudflared_bin.exists() or cloudflared_bin.stat().st_size < 1_000_000:
    if cloudflared_bin.exists():
        cloudflared_bin.unlink()
        print("cloudflared: stale file removed (size < 1MB)")
    print("Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared_bin}
    !chmod +x {cloudflared_bin}
    import os as _os; size_kb = _os.path.getsize(str(cloudflared_bin)) / 1024
    print(f"cloudflared: done ({size_kb:.0f} KB)")
else:
    size_kb = cloudflared_bin.stat().st_size / 1024
    print(f"cloudflared: cached ({size_kb:.0f} KB)")

# ---- GarmentRec: Clone repo + download assets + model weights ----
GARMENT_REC_DIR = WD / "GarmentRec"
if not (GARMENT_REC_DIR / "code").exists():
    print("Cloning GarmentRec repo...")
    !git clone --depth=1 https://github.com/worryDes/GarmentRec.git {GARMENT_REC_DIR}
else:
    print("GarmentRec repo: cached")

# ---- GarmentRec: Robust Asset Extraction ----
GARMENT_REC_DIR = WD / "GarmentRec"
_data_dir = GARMENT_REC_DIR / "data"
_tmps = _data_dir / "tmps"
midpairs_path = _data_dir / "midpairs.pkl"

# Integrity check: pickles + tmps folder (tmps from 7z contains PCA/UV for template gen)
dense_midpairs_path_c4 = _data_dir / "dense_midpairs.pkl"
assets_valid = (
    midpairs_path.exists() and midpairs_path.stat().st_size > 100
    and dense_midpairs_path_c4.exists() and dense_midpairs_path_c4.stat().st_size > 100
    and _tmps.exists()
) 

if not assets_valid:
    print("Downloading/Extracting GarmentRec assets...")
    !apt-get install -y -qq p7zip-full > /dev/null 2>&1
    if not Path("/tmp/garmentrec_assets.7z").exists():
        !gdown 1PTbfEMchwgHpaL3y8Gm1__sbzFLqfoYj -O /tmp/garmentrec_assets.7z --quiet
    
    # Clean temp extraction
    if os.path.exists("/tmp/garmentrec_assets_tmp"): shutil.rmtree("/tmp/garmentrec_assets_tmp")
    os.makedirs("/tmp/garmentrec_assets_tmp")
    
    !7z x /tmp/garmentrec_assets.7z -o/tmp/garmentrec_assets_tmp -y > /dev/null 2>&1
    import glob
    print("Robustly locating extracted assets...")
    found_mid = glob.glob("/tmp/garmentrec_assets_tmp/**/midpairs.pkl", recursive=True)
    if found_mid:
        data_root = Path(found_mid[0]).parent
        print(f"Detected asset root: {data_root}")
        (GARMENT_REC_DIR / "data").mkdir(parents=True, exist_ok=True)
        !cp -r {data_root}/* {GARMENT_REC_DIR}/data/
    
    # Locate data root (where midpairs.pkl is)
    found = glob.glob("/tmp/garmentrec_assets_tmp/**/midpairs.pkl", recursive=True)
    if found:
        src_root = Path(found[0]).parent
        print(f"Detected asset root: {src_root}")
        _data_dir.mkdir(parents=True, exist_ok=True)
        # Copy everything safely
        for item in os.listdir(src_root):
            s = src_root / item
            d = _data_dir / item
            if s.is_dir():
                if d.exists(): shutil.rmtree(d)
                shutil.copytree(s, d)
            else:
                shutil.copy2(s, d)
        print("Assets copied successfully.")
    else:
        print("CRITICAL: Could not find midpairs.pkl in extracted archive!")
else:
    print("GarmentRec assets: cached and verified.")


# Ensure tmps folder is in the right place (7z may extract to wrong path)
tmps_target = GARMENT_REC_DIR / "data" / "tmps"
if not tmps_target.exists():
    # Search for tmps folder anywhere under GARMENT_REC_DIR
    for candidate in [
        GARMENT_REC_DIR / "tmps",
        GARMENT_REC_DIR / "models" / "tmps",
        GARMENT_REC_DIR / "archive" / "tmps",
        GARMENT_REC_DIR / "archive" / "data" / "tmps",
        GARMENT_REC_DIR / "archive" / "data" / "data" / "tmps",
        Path("/tmp/garmentrec_assets_tmp") / "data" / "tmps",
        Path("/tmp/garmentrec_assets_tmp") / "tmps",
    ]:
        if candidate.exists():
            tmps_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(candidate), str(tmps_target))
            print(f"Moved tmps from {candidate} to {tmps_target}")
            break

# GarmentRec model weights — multi-layer fallback: Kaggle Dataset → wget → requests
model_dir = GARMENT_REC_DIR / "models" / "mrf_0.1_shading_0.1"
model_pth = model_dir / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"

# Remove stale/broken files (anything under 100MB is invalid)
if model_pth.exists() and model_pth.stat().st_size < 100 * 1024 * 1024:
    model_pth.unlink()
    print(f"Removed stale file ({model_pth.stat().st_size / 1024:.0f} KB), re-downloading...")

if not model_pth.exists():
    dl_ok = False
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Layer 1: Kaggle Dataset (GCP-local, no Xet, instant)
    if kaggle_path:
        src = Path(kaggle_path) / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"
        if src.exists():
            shutil.copy(src, model_pth)
            print(f"Model weights: Kaggle Dataset ({model_pth.stat().st_size/(1024*1024):.0f} MB)")
            dl_ok = True
    
    # Layer 2: HF Hub via wget/requests (bypasses Xet Rust protocol)
    if not dl_ok:
        import torch as _torch
        model_url = ("https://huggingface.co/jacobthankgod4/smpl-model-garmentrec/"
                     "resolve/main/mrf_0.1_shading_0.1_pca64_ep100_bth0.pth?download=1")
        dl_ok = robust_download(model_url, model_pth, expected_mb=1100, desc="Model weights (~1.1GB)")
        # Post-download integrity check: try loading with torch
        if dl_ok:
            try:
                _chk = _torch.load(str(model_pth), map_location='cpu', weights_only=True)
                del _chk
                print(f"  Integrity: OK")
            except Exception as _e:
                print(f"  Integrity FAILED ({_e}), removing corrupt file...")
                model_pth.unlink(missing_ok=True)
                dl_ok = False

    # Layer 3: hf_hub_download (different CDN path, may bypass Xet 403)
    if not dl_ok:
        print("  Trying hf_hub_download (Layer 3)...")
        from huggingface_hub import hf_hub_download
        import subprocess as _sp, sys as _sys
        _env = {**{k: v for k, v in _os.environ.items()}, 'HF_HUB_DISABLE_XET': '1'}
        try:
            _downloaded = hf_hub_download(
                repo_id='jacobthankgod4/smpl-model-garmentrec',
                filename='mrf_0.1_shading_0.1_pca64_ep100_bth0.pth',
                local_dir=str(model_dir),
                local_dir_use_symlinks=False,
            )
            if Path(_downloaded).exists() and Path(_downloaded).stat().st_size > 100 * 1024 * 1024:
                print(f"  hf_hub_download: OK ({Path(_downloaded).stat().st_size/(1024*1024):.0f} MB)")
                dl_ok = True
        except Exception as _e:
            print(f"  hf_hub_download failed: {_e}")
    
    if dl_ok:
        print(f"GarmentRec model: done ({model_pth.stat().st_size/(1024*1024):.0f} MB)")
    else:
        print("*** GARMENTREC MODEL WEIGHTS MISSING — reconstruction will fail ***")
else:
    size_mb = model_pth.stat().st_size / (1024*1024)
    print(f"GarmentRec model: cached ({size_mb:.0f} MB)")

# Final checks
tmps_target = GARMENT_REC_DIR / "data" / "tmps"
GARMENTS = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt", "Shorts", "Pants"]
if tmps_target.exists():
    all_ok = True
    for g in GARMENTS:
        obj = tmps_target / g / "garment_tmp.obj"
        ok = obj.exists() and obj.stat().st_size > 0
        if not ok:
            print(f"  *** {g}: MISSING garment_tmp.obj ***")
            all_ok = False
    if all_ok:
        print(f"tmps folder: OK ({len(list(tmps_target.iterdir()))} garment types, files verified)")
    else:
        print(f"\n*** tmps INCOMPLETE — see above. Re-run or check /tmp/garmentrec_assets_tmp/ ***")
else:
    print(f"\n*** tmps MISSING *** Expected at {tmps_target}")
    print("Check /tmp/garmentrec_assets_tmp/ for raw 7z contents.")

midpairs = GARMENT_REC_DIR / "data" / "midpairs.pkl"
dense_midpairs = GARMENT_REC_DIR / "data" / "dense_midpairs.pkl"
if midpairs.exists() and midpairs.stat().st_size > 100:
    print(f"midpairs.pkl: OK ({midpairs.stat().st_size / 1024:.1f} KB)")
else:
    print(f"*** midpairs.pkl MISSING at {midpairs} ***")
if dense_midpairs.exists():
    print(f"dense_midpairs.pkl: OK ({dense_midpairs.stat().st_size / 1024:.1f} KB)")
else:
    print(f"*** dense_midpairs.pkl MISSING at {dense_midpairs} ***")

# Fallback: download midpairs files from GitHub repo (they are git-tracked, not in 7z archive)
import urllib.request
for _pkl_name in ["midpairs.pkl", "dense_midpairs.pkl"]:
    _pkl_path = GARMENT_REC_DIR / "data" / _pkl_name
    if not _pkl_path.exists() or _pkl_path.stat().st_size < 100:
        _url = f"https://raw.githubusercontent.com/worryDes/GarmentRec/main/data/{_pkl_name}"
        print(f"Downloading {_pkl_name} from GitHub...")
        try:
            urllib.request.urlretrieve(_url, _pkl_path)
            print(f"  -> {_pkl_path} ({_pkl_path.stat().st_size / 1024:.0f} KB)")
        except Exception as _e:
            print(f"  *** FALLBACK FAILED for {_pkl_name}: {_e} ***")

# Check model weights
if not model_pth.exists() or model_pth.stat().st_size < 100 * 1024 * 1024:
    print(f"\n*** MODEL WEIGHTS {'MISSING' if not model_pth.exists() else 'TOO SMALL'} ***")
    print(f"Expected: {model_pth}")
else:
    print(f"Model weights: OK ({model_pth.stat().st_size / (1024*1024):.0f} MB)")

# SMPL model: multi-layer fallback
smpl_dir = GARMENT_REC_DIR / "smpl_pytorch" / "model"
smpl_dir.mkdir(parents=True, exist_ok=True)
smpl_path = smpl_dir / "neutral_smpl_with_cocoplus_reg.txt"
smpl_pkl = WD / "neutral_smpl_with_cocoplus_reg.pkl"
if not smpl_path.exists():
    smpl_ok = False
    # Layer 1: Kaggle Dataset
    if kaggle_path:
        src = Path(kaggle_path) / "neutral_smpl_with_cocoplus_reg.txt"
        if src.exists():
            shutil.copy(src, smpl_path)
            print(f"SMPL model: Kaggle Dataset")
            smpl_ok = True
    # Layer 2: HF Hub via wget/requests
    if not smpl_ok:
        smpl_url = ("https://huggingface.co/jacobthankgod4/smpl-model-garmentrec/"
                    "resolve/main/neutral_smpl_with_cocoplus_reg.txt")
        smpl_ok = robust_download(smpl_url, smpl_path, desc="SMPL model")
    if smpl_ok:
        print(f"SMPL model: OK ({smpl_path.stat().st_size/1024:.0f} KB)")
    else:
        print("*** SMPL MODEL MISSING ***")

if not smpl_path.exists() and smpl_pkl.exists():
    print("Converting SMPL .pkl to .txt (JSON)...")
    import pickle, json
    with open(smpl_pkl, 'rb') as f:
        pkl_data = pickle.load(f, encoding='latin1')
    # Convert chumpy/numpy/sparse → plain lists
    def to_serializable(v):
        if hasattr(v, 'r'):
            return v.r.tolist()
        if hasattr(v, 'toarray'):
            return v.toarray().tolist()
        if hasattr(v, 'tolist'):
            return v.tolist()
        if isinstance(v, (list, tuple)):
            return list(v)
        return v
    model_json = {k: to_serializable(v) for k, v in pkl_data.items()}
    with open(smpl_path, 'w') as f:
        json.dump(model_json, f)
    size_mb = smpl_path.stat().st_size / (1024*1024)
    print(f"SMPL model saved ({size_mb:.0f} MB)")

if not smpl_path.exists():
    print("\n*** SMPL MODEL NEEDED ***")
    print("Place neutral_smpl_with_cocoplus_reg.pkl or .txt at:")
    print(f"  {smpl_path}")
    print("Without this file, GarmentRec will fail at startup.\n")
else:
    size_mb = smpl_path.stat().st_size / (1024*1024)
    print(f"SMPL model ready ({size_mb:.0f} MB)")

# Patch deprecated numpy aliases in GarmentRec (NumPy 2.x removed np.float)
!sed -i 's/np\.float/float/g' {GARMENT_REC_DIR}/smpl_pytorch/SMPL.py
!sed -i 's/np\.float/float/g' {GARMENT_REC_DIR}/smpl_pytorch/util.py
# Patch SMPL J_regressor: transpose (24,6890) -> (6890,24) so matmul works
!sed -i "s/np_J_regressor = np.array(model\['J_regressor'\], dtype = np.float)/np_J_regressor = np.array(model['J_regressor'], dtype = np.float).T/" {GARMENT_REC_DIR}/smpl_pytorch/SMPL.py
print("Patched GarmentRec: np.float -> float (NumPy 2.x compat), J_regressor transposed")

# ---- Verify pymeshlab (fallback install if kernel restart broke it) ----
try:
    import pymeshlab
    print("pymeshlab: already installed")
except ImportError:
    print("pymeshlab: not found, installing...")
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pymeshlab"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    import pymeshlab
    print("pymeshlab: installed")

# ---- Verify pymeshlab (fallback install if kernel restart broke it) ----
try:
    import pymeshlab
    print("pymeshlab: available")
except ImportError:
    print("pymeshlab: not found, (re)installing...")
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pymeshlab"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    import pymeshlab
    print("pymeshlab: (re)installed successfully")

# ---- Generate missing garment templates (Robust) ----
import numpy as np
import pickle
from collections import deque

tmps_dir = GARMENT_REC_DIR / "data" / "tmps"
GARMENTS_META = [
    ("T-shirt", 1954), ("front_open_T-shirt", 1954),
    ("Shirt", 2468), ("front_open_Shirt", 2468),
    ("Shorts", 678), ("Pants", 1180),
]

for gtype, vnum in GARMENTS_META:
    try:
        gar_dir = tmps_dir / gtype
        obj_path = gar_dir / "garment_tmp.obj"
        dense_obj_path = gar_dir / "garment_tmp_subdivide_uv_new.obj"
        spiral_path = gar_dir / "spiral_indices_2.npy"
        pca_path = gar_dir / "pca_data_64.npz"
        tex_uv_path = gar_dir / "tex_uv.pkl"

        if obj_path.exists() and dense_obj_path.exists() and spiral_path.exists() and obj_path.stat().st_size > 0:
            print(f"{gtype}: templates cached")
            continue
            
        if not pca_path.exists() or not tex_uv_path.exists():
            print(f"*** {gtype}: SKIPPING, base assets missing (PCA/UV) ***")
            continue

        print(f"Processing {gtype}...")
        import numpy as _np; pca = _np.load(str(pca_path))
        verts = pca["mean"].reshape(-1, 3)
        tex_uv = pickle.load(open(str(tex_uv_path), "rb"))
        
        # Determine UV source (support both tensor and ndarray)
        fu = tex_uv["faces_uvs"]
        vu = tex_uv["verts_uvs"]
        faces_uvs = fu.numpy() if hasattr(fu, "numpy") else fu
        verts_uvs = vu.numpy() if hasattr(vu, "numpy") else vu

        # Write obj
        with open(str(obj_path), 'w') as f:
            for v in verts: f.write(f'v {v[0]:.8f} {v[1]:.8f} {v[2]:.8f}\n')
            for vt in verts_uvs: f.write(f'vt {vt[0]:.8f} {vt[1]:.8f}\n')
            for face in faces_uvs: f.write(f'f {face[0]+1}/{face[0]+1} {face[1]+1}/{face[1]+1} {face[2]+1}/{face[2]+1}\n')

        # Subdivision
        import pymeshlab
        ms = pymeshlab.MeshSet()
        ms.load_new_mesh(str(obj_path))
        _Threshold = getattr(pymeshlab, 'PercentageValue', None) or getattr(pymeshlab, 'Percentage', None)
        ms.meshing_surface_subdivision_loop(loopweight=0, iterations=2, threshold=_Threshold(0) if _Threshold else 0)
        m = ms.current_mesh()
        dense_verts = m.vertex_matrix()
        dense_faces = m.face_matrix()
        with open(str(dense_obj_path), 'w') as f:
            for v in dense_verts: f.write(f'v {v[0]:.8f} {v[1]:.8f} {v[2]:.8f}\n')
            if m.has_vertex_tex_coord():
                vt = m.vertex_tex_coord_matrix()
                for uv in vt: f.write(f'vt {uv[0]:.8f} {uv[1]:.8f}\n')
                for face in dense_faces:
                    f.write(f'f {face[0]+1}/{face[0]+1} {face[1]+1}/{face[1]+1} {face[2]+1}/{face[2]+1}\n')
            else:
                for face in dense_faces:
                    f.write(f'f {face[0]+1} {face[1]+1} {face[2]+1}\n')

        # Spiral (using in-memory verts/faces_uvs instead of openmesh)
        V = verts.shape[0]
        faces_for_adj = np.array(faces_uvs, dtype=int)
        adj = {i: set() for i in range(V)}
        for face in faces_for_adj:
            for i in range(3):
                for j in range(i+1, 3):
                    v0, v1 = face[i], face[j]
                    adj[v0].add(v1); adj[v1].add(v0)
        max_neighbors = 20
        spiral = np.zeros((V, max_neighbors), dtype=np.int64)
        for v in range(V):
            seq = [v]; visited = {v}
            q = deque([(nbr, 1) for nbr in sorted(adj[v])])
            visited.update(adj[v]); seq.extend(sorted(adj[v]))
            while len(seq) < max_neighbors and q:
                cur, dist = q.popleft()
                for nbr in sorted(adj[cur]):
                    if nbr not in visited and len(seq) < max_neighbors:
                        visited.add(nbr); seq.append(nbr); q.append((nbr, dist + 1))
            while len(seq) < max_neighbors: seq.append(v)
            spiral[v] = seq[:max_neighbors]
        np.save(str(spiral_path), spiral[np.newaxis, :, :])
        print(f"{gtype}: Done (Spiral: {spiral.shape[0]}x{max_neighbors})")
    except Exception as e:
        print(f"*** {gtype}: FAILED -> {e} ***")

print("GarmentRec templates: processing finished")


# ---- GarmentGPT: Clone repo + download checkpoints ----
GARMENT_GPT_DIR = WD / "Garment-GPT"
if not (GARMENT_GPT_DIR / "main.py").exists():
    print("Cloning Garment-GPT repo...")
    !git clone --depth=1 https://github.com/ChimerAI-MMLab/Garment-GPT.git {GARMENT_GPT_DIR}
else:
    print("Garment-GPT repo: cached")

# LLaMA-Factory (required by GarmentGPT main.py imports)
LLAMA_FACTORY = WD / "LLaMA-Factory"
if not (LLAMA_FACTORY / "setup.py").exists():
    print("Cloning LLaMA-Factory...")
    !git clone --depth=1 https://github.com/hiyouga/LLaMA-Factory.git {LLAMA_FACTORY}
    !pip install -q -e {LLAMA_FACTORY}
else:
    print("LLaMA-Factory: cached")

# vLLM shim (P100 sm_60 can't run real vLLM — uses transformers + int8 quantization)
# Use base64 to avoid escaping issues with inline triple-quoted strings
VLLM_SHIM = WD / "vllm_shim.py"
import base64
VLLM_SHIM.write_bytes(base64.b64decode(
    "IiIidkxMTSBjb21wYXRpYmlsaXR5IGxheWVyIHVzaW5nIEh1Z2dpbmdGYWNlIFRyYW5zZm9ybWVycyArIGJpdHNhbmRieXRlcyBpbnQ4LiIiIgppbXBvcnQgc3lzLCBvcywgdHlwZXMsIHRvcmNoCgpfdmxsbV9yZWFsID0gTm9uZQp0cnk6CiAgICBpbXBvcnQgdmxsbSBhcyBfdmxsbV9yZWFsCiAgICBfSGFzVkxMTSA9IFRydWUKICAgIHByaW50KGYiW3ZsbG0tc2hpbV0gUmVhbCB2bGxtIGltcG9ydGVkOiBDVURBIGNhcHMge3RvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KCl9IikKZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgX0hhc1ZMTE0gPSBGYWxzZQogICAgcHJpbnQoZiJbdmxsbS1zaGltXSBSZWFsIHZsbG0gbm90IGF2YWlsYWJsZToge2V9IikKCgpjbGFzcyBfSEZfTExNOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsLCAqKmt3YXJncyk6CiAgICAgICAgc2VsZi5fbW9kZWwgPSBOb25lCiAgICAgICAgc2VsZi5fcHJvY2Vzc29yID0gTm9uZQogICAgICAgIHNlbGYuX3JlYWwgPSBOb25lCiAgICAgICAgaWYgX3ZsbG1fcmVhbCBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5fcmVhbCA9IF92bGxtX3JlYWwuTExNKG1vZGVsPW1vZGVsLCAqKmt3YXJncykKICAgICAgICAgICAgICAgIHByaW50KCJbdmxsbS1zaGltXSBVc2luZyBSRUFMIHZMTE0gZW5naW5lIikKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBwcmludChmIlt2bGxtLXNoaW1dIFJlYWwgdkxMTSBpbml0IGZhaWxlZDoge2V9IikKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Byb2Nlc3NvciwgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEJpdHNBbmRCeXRlc0NvbmZpZwogICAgICAgIHByaW50KGYiW3ZsbG0tc2hpbV0gTG9hZGluZyBtb2RlbCBmcm9tIHttb2RlbH0gd2l0aCBpbnQ4IHF1YW50aXphdGlvbi4uLiIpCiAgICAgICAgYm5iX2NvbmZpZyA9IEJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzhiaXQ9VHJ1ZSwgbGxtX2ludDhfdGhyZXNob2xkPTYuMCkKICAgICAgICBzZWxmLl9wcm9jZXNzb3IgPSBBdXRvUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChtb2RlbCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgICAgICBzZWxmLl9tb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgbW9kZWwsIHF1YW50aXphdGlvbl9jb25maWc9Ym5iX2NvbmZpZywgZGV2aWNlX21hcD0iYXV0byIsCiAgICAgICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsIHRvcmNoX2R0eXBlPXRvcmNoLmZsb2F0MTYsCiAgICAgICAgKQogICAgICAgIHNlbGYuX21vZGVsLmV2YWwoKQogICAgICAgIHByaW50KGYiW3ZsbG0tc2hpbV0gTW9kZWwgbG9hZGVkIG9uIHtzZWxmLl9tb2RlbC5kZXZpY2V9IHdpdGggaW50OCIpCgogICAgZGVmIGdlbmVyYXRlKHNlbGYsIGlucHV0c19saXN0LCBzYW1wbGluZ19wYXJhbXMpOgogICAgICAgIGlmIHNlbGYuX3JlYWwgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9yZWFsLmdlbmVyYXRlKGlucHV0c19saXN0LCBzYW1wbGluZ19wYXJhbXMpCiAgICAgICAgcmVzdWx0cyA9IFtdCiAgICAgICAgZm9yIGlucCBpbiBpbnB1dHNfbGlzdDoKICAgICAgICAgICAgcHJvbXB0ID0gaW5wLmdldCgicHJvbXB0IiwgIiIpCiAgICAgICAgICAgIGltYWdlcyA9IGlucC5nZXQoIm11bHRpX21vZGFsX2RhdGEiLCB7fSkuZ2V0KCJpbWFnZSIsIFtdKQogICAgICAgICAgICBpZiBpbWFnZXM6CiAgICAgICAgICAgICAgICBwcm9jID0gc2VsZi5fcHJvY2Vzc29yKHRleHQ9cHJvbXB0LCBpbWFnZXM9aW1hZ2VzWzBdLCByZXR1cm5fdGVuc29ycz0icHQiKS50byhzZWxmLl9tb2RlbC5kZXZpY2UpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBwcm9jID0gc2VsZi5fcHJvY2Vzc29yKHRleHQ9cHJvbXB0LCByZXR1cm5fdGVuc29ycz0icHQiKS50byhzZWxmLl9tb2RlbC5kZXZpY2UpCiAgICAgICAgICAgIG10ID0gZ2V0YXR0cihzYW1wbGluZ19wYXJhbXMsICdtYXhfdG9rZW5zJywgNDA5NikKICAgICAgICAgICAgdHAgPSBnZXRhdHRyKHNhbXBsaW5nX3BhcmFtcywgJ3RlbXBlcmF0dXJlJywgMC4xKQogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIG91dCA9IHNlbGYuX21vZGVsLmdlbmVyYXRlKCoqcHJvYywgbWF4X25ld190b2tlbnM9bWluKG10LCA0MDk2KSwKICAgICAgICAgICAgICAgICAgICBkb19zYW1wbGU9dHAgPiAwLCB0ZW1wZXJhdHVyZT1tYXgodHAsIDAuMDEpLCB0b3BfcD0wLjkpCiAgICAgICAgICAgIGdlbiA9IHNlbGYuX3Byb2Nlc3Nvci5kZWNvZGUob3V0WzBdLCBza2lwX3NwZWNpYWxfdG9rZW5zPUZhbHNlKQogICAgICAgICAgICBjbGFzcyBfTzoKICAgICAgICAgICAgICAgIGRlZiBfX2luaXRfXyhzLCB0ZXh0KTogcy50ZXh0ID0gdGV4dAogICAgICAgICAgICBjbGFzcyBfUk86CiAgICAgICAgICAgICAgICBkZWYgX19pbml0X18ocywgb3V0cHV0cyk6IHMub3V0cHV0cyA9IG91dHB1dHMKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoX1JPKG91dHB1dHM9W19PKHRleHQ9Z2VuKV0pKQogICAgICAgIHJldHVybiByZXN1bHRzCgoKY2xhc3MgX0hGX1NhbXBsaW5nUGFyYW1zOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRlbXBlcmF0dXJlPTAuMSwgbWF4X3Rva2Vucz00MDk2LCBza2lwX3NwZWNpYWxfdG9rZW5zPUZhbHNlLAogICAgICAgICAgICAgICAgIHNlZWQ9NDIsIHN0b3BfdG9rZW5faWRzPU5vbmUpOgogICAgICAgIHNlbGYudGVtcGVyYXR1cmUgPSB0ZW1wZXJhdHVyZQogICAgICAgIHNlbGYubWF4X3Rva2VucyA9IG1heF90b2tlbnMKICAgICAgICBzZWxmLnNraXBfc3BlY2lhbF90b2tlbnMgPSBza2lwX3NwZWNpYWxfdG9rZW5zCiAgICAgICAgc2VsZi5zZWVkID0gc2VlZAogICAgICAgIHNlbGYuc3RvcF90b2tlbl9pZHMgPSBzdG9wX3Rva2VuX2lkcyBvciBbXQoKCnZsbG1fbW9kID0gdHlwZXMuTW9kdWxlVHlwZSgidmxsbSIpCnZsbG1fbW9kLkxMTSA9IF9IRl9MTE0KdmxsbV9tb2QuU2FtcGxpbmdQYXJhbXMgPSBfSEZfU2FtcGxpbmdQYXJhbXMKc3lzLm1vZHVsZXNbInZsbG0iXSA9IHZsbG1fbW9kCnByaW50KGYiW3ZsbG0tc2hpbV0gSW5zdGFsbGVkIChyZWFsIHZsbG06IHtfSGFzVkxMTX0pIikK"))
sys.path.insert(0, str(WD))  # ensure shim is importable
import vllm_shim  # triggers sys.modules shadow for 'vllm'
print("vLLM shim activated")

# GarmentGPT checkpoints (inference-only: skip optimizer states, download ~14GB)
CHECKPOINTS_DIR = GARMENT_GPT_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Inference files only: 3 model shards (~14GB) + config/tokenizer + codec models (~642MB)
# SKIP: global_step12844/ (~80GB optimizer states), rng_state_*.pth, scheduler.pt, training_args.bin
gpt_files = [
    "vlm/checkpoint-12844/model-00001-of-00003.safetensors",  # 4.98 GB
    "vlm/checkpoint-12844/model-00002-of-00003.safetensors",  # 4.99 GB
    "vlm/checkpoint-12844/model-00003-of-00003.safetensors",  # 4.19 GB
    "vlm/checkpoint-12844/model.safetensors.index.json",
    "vlm/checkpoint-12844/config.json",
    "vlm/checkpoint-12844/generation_config.json",
    "vlm/checkpoint-12844/preprocessor_config.json",
    "vlm/checkpoint-12844/processor_config.json",
    "vlm/checkpoint-12844/tokenizer.json",
    "vlm/checkpoint-12844/tokenizer.model",
    "vlm/checkpoint-12844/tokenizer_config.json",
    "vlm/checkpoint-12844/special_tokens_map.json",
    "vlm/checkpoint-12844/added_tokens.json",
    "vlm/checkpoint-12844/chat_template.json",
    "vqvae/best_model.pth",     # 622 MB (edge codec)
    "vqvae/rt_vqvae.pth",       # 20 MB (RT codec)
]

from concurrent.futures import ThreadPoolExecutor, as_completed

HF_AUTH = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}
GPT_BASE = "https://huggingface.co/ChimerAI/GarmentGPT/resolve/main"

# Separate large safetensors (parallel) from small config/codec files (sequential)
large_files = [f for f in gpt_files if f.endswith(".safetensors") and "model-" in f]
small_files = [f for f in gpt_files if f not in large_files]

def download_gpt_file(f):
    dest = CHECKPOINTS_DIR / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return True
    url = f"{GPT_BASE}/{f}?download=1"
    if robust_download(url, dest, extra_headers=HF_AUTH, desc=f):
        return True
    # Layer 2: hf_hub_download (different CDN path, may bypass Xet 403)
    print(f"  {f}: fallback to hf_hub_download...")
    try:
        hf_hub_download(
            repo_id="ChimerAI/GarmentGPT",
            filename=f,
            local_dir=str(CHECKPOINTS_DIR),
            local_dir_use_symlinks=False,
            force_download=True,
        )
        if dest.exists() and dest.stat().st_size > 0:
            return True
    except Exception as _e2:
        print(f"  {f}: hf_hub_download fallback failed: {_e2}")
    return False

need_download = not (CHECKPOINTS_DIR / "vqvae" / "best_model.pth").exists()
if need_download:
    # Parallel: 3 large safetensors (4.98 + 4.99 + 4.19 = ~14.2 GB, ~3x speedup)
    print("Downloading 3 large model shards in parallel...")
    with ThreadPoolExecutor(max_workers=3) as pool:
        fut_to_file = {pool.submit(download_gpt_file, f): f for f in large_files}
        for fut in as_completed(fut_to_file):
            f = fut_to_file[fut]
            if fut.result():
                size_mb = (CHECKPOINTS_DIR / f).stat().st_size / (1024*1024)
                print(f"  {f}: {size_mb:.0f} MB")
            else:
                print(f"  *** {f}: FAILED ***")
    
    # Sequential: smaller config/tokenizer/codec files
    for f in small_files:
        download_gpt_file(f)
    
    print("GarmentGPT checkpoints: done (~14.6GB model + 642MB codecs)")
else:
    print("GarmentGPT checkpoints: cached")

# ---- Kaggle Dataset auto-upload (cache for future runs) ----
try:
    _kaggle_dir = Path("/tmp/kaggle_dataset_upload")
    if _kaggle_dir.exists(): shutil.rmtree(_kaggle_dir)
    _kaggle_dir.mkdir(parents=True)
    _upload_files = {
        model_pth: "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth",
        smpl_path: "neutral_smpl_with_cocoplus_reg.txt",
    }
    for src, name in _upload_files.items():
        if src.exists():
            (Path(_kaggle_dir) / name).symlink_to(src)
    import json as _json
    with open(_kaggle_dir / "dataset-metadata.json", "w") as _f:
        _json.dump({
            "title": "Korra Garment Weights",
            "id": "jacobthankgod/korra-garment-weights",
            "description": "Pretrained weights for GarmentRec + SMPL model, auto-uploaded from Kaggle notebook.",
            "licenses": [{"name": "MIT"}],
        }, _f)
    import subprocess as _sp
    _r = _sp.run(["kaggle", "datasets", "create", "-p", str(_kaggle_dir), "--dir-mode", "tar", "--quiet"],
                capture_output=True, text=True, timeout=120)
    if _r.returncode == 0:
        print(f"Kaggle Dataset '{KAGGLE_DATASET}': uploaded for future runs")
    elif "already exists" in (_r.stderr or ""):
        _r2 = _sp.run(["kaggle", "datasets", "version", "-p", str(_kaggle_dir), "--dir-mode", "tar", "--quiet"],
                    capture_output=True, text=True, timeout=120)
        if _r2.returncode == 0:
            print(f"Kaggle Dataset '{KAGGLE_DATASET}': version updated")
        else:
            print(f"Kaggle Dataset version update failed (non-fatal): {_r2.stderr[:200]}")
    else:
        print(f"Kaggle Dataset upload failed (non-fatal): {_r.stderr[:200]}")
except Exception as _e:
    print(f"Kaggle Dataset auto-upload skipped (non-fatal): {_e}")

# ---- SAM2 weights ----
sam2_dir = WEIGHTS_DIR / "sam2"
sam2_dir.mkdir(exist_ok=True)
if not (sam2_dir / "sam2_hiera_large.pt").exists():
    print("Downloading SAM2 weights...")
    !wget -q -O {sam2_dir}/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
else:
    print("SAM2: cached")

# ---- GarmentCode ----
garmentcode_dir = WEIGHTS_DIR / "garmentcode"
if not (garmentcode_dir / "repo").exists():
    print("Cloning GarmentCode...")
    !git clone --depth=1 https://github.com/maria-korosteleva/GarmentCode.git {garmentcode_dir}/repo
else:
    print("GarmentCode: cached")

print("\nStorage used:")
!du -sh {WEIGHTS_DIR}/* 2>/dev/null || true
!du -sh {GARMENT_REC_DIR} 2>/dev/null || true
!du -sh {GARMENT_GPT_DIR} 2>/dev/null || true


## Cell 3: Setup HuggingFace Token

In [ ]:
from huggingface_hub import login
login(token="")
print("Logged in to HuggingFace Hub")


## Cell 4: Write API Server

In [ ]:
code = r'''"""
Garment Reconstruction API Server (Async + Split Pipeline + SAM2 Fix)
Runs on Kaggle GPU backend, exposed via Cloudflare Tunnel.

Pipeline: Image -> 3D Mesh + Sewing Pattern

Endpoints:
  POST /api/v1/reconstruct     -> async job (returns job_id, processes in background)
  POST /api/v1/segment         -> Step 1: rembg + SAM2 (~20s)
  POST /api/v1/mesh            -> Step 2: GarmentRec (~50s)
  POST /api/v1/pattern         -> Step 3: GarmentGPT (~50s)
  POST /api/v1/callback        -> receives result URL from processing
  GET  /api/v1/job/{job_id}    -> poll job status
  GET  /health                 -> health check
  GET  /debug/error            -> last error traceback
"""

# ---- vLLM compatibility layer (SHIM) ----
import sys, types, torch

def _activate_vllm_shim():
    import importlib
    _orig_find_spec = importlib.util.find_spec
    def _patched_find_spec(name, *args, **kwargs):
        if name == 'vllm':
            return None
        return _orig_find_spec(name, *args, **kwargs)
    importlib.util.find_spec = _patched_find_spec

    class _HF_LLM:
        def __init__(self, model, **kwargs):
            from transformers import AutoProcessor, AutoConfig
            import torch

            use_8bit = False
            gpu_label = "cpu"
            if torch.cuda.is_available():
                cap = torch.cuda.get_device_capability()
                gpu_label = f"sm_{cap[0]}{cap[1]}"
                if cap >= (7, 0):
                    use_8bit = True

            print(f'[vllm-shim] Loading model {model} on {gpu_label}...')
            self._processor = AutoProcessor.from_pretrained(model, trust_remote_code=True)
            cfg_kwargs = dict(kwargs)
            cfg_kwargs.setdefault('trust_remote_code', True)
            cfg = AutoConfig.from_pretrained(model, **cfg_kwargs)
            if hasattr(cfg, 'vision_config'):
                try:
                    from transformers import AutoModelForVision2Seq
                    model_class = AutoModelForVision2Seq
                except ImportError:
                    from transformers import LlavaForConditionalGeneration
                    model_class = LlavaForConditionalGeneration
            else:
                from transformers import AutoModelForCausalLM
                model_class = AutoModelForCausalLM

            load_kwargs = dict(
                device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
            )
            if use_8bit:
                from transformers import BitsAndBytesConfig
                load_kwargs['quantization_config'] = BitsAndBytesConfig(
                    load_in_8bit=True, llm_int8_threshold=6.0,
                )

            self._model = model_class.from_pretrained(model, **load_kwargs)
            self._model.eval()
            dtype_label = "int8" if use_8bit else "fp16"
            print(f'[vllm-shim] Model loaded on {self._model.device} in {dtype_label}')

        def generate(self, inputs_list, sampling_params):
            results = []
            for inp in inputs_list:
                prompt = inp.get('prompt', '')
                images = inp.get('multi_modal_data', {}).get('image', [])
                if images:
                    proc = self._processor(text=prompt, images=images[0], return_tensors='pt').to(self._model.device)
                else:
                    proc = self._processor(text=prompt, return_tensors='pt').to(self._model.device)
                mt = getattr(sampling_params, 'max_tokens', 4096)
                tp = getattr(sampling_params, 'temperature', 0.1)
                with torch.no_grad():
                    out = self._model.generate(**proc, max_new_tokens=min(mt, 4096),
                        do_sample=tp > 0, temperature=max(tp, 0.01), top_p=0.9)
                gen = self._processor.decode(out[0], skip_special_tokens=False)
                class _O:
                    def __init__(self, text): self.text = text
                class _RO:
                    def __init__(self, outputs): self.outputs = outputs
                results.append(_RO(outputs=[_O(text=gen)]))
            return results

    class _HF_SamplingParams:
        def __init__(self, temperature=0.1, max_tokens=4096, skip_special_tokens=False, seed=42, stop_token_ids=None):
            self.temperature = temperature
            self.max_tokens = max_tokens
            self.skip_special_tokens = skip_special_tokens
            self.seed = seed
            self.stop_token_ids = stop_token_ids or []

    vllm_mod = types.ModuleType('vllm')
    vllm_mod.LLM = _HF_LLM
    vllm_mod.SamplingParams = _HF_SamplingParams
    sys.modules['vllm'] = vllm_mod
    print('[vllm-shim] vLLM compatibility shim installed (transformers + int8 backend)')

_activate_vllm_shim()

import os
import io
import json
import time
import uuid
import zipfile
import tempfile
import logging
import gc
import asyncio
import threading
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import asynccontextmanager

import numpy as np
import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("garment-api")

WORKING_DIR = Path("/kaggle/working")
WEIGHTS_DIR = WORKING_DIR / "weights"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_DEVICE = DEVICE
CPU_DEVICE = "cpu"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- repo paths (cloned by Cell 2) ----
GARMENT_REC_DIR = WORKING_DIR / "GarmentRec"
GARMENT_GPT_DIR = WORKING_DIR / "Garment-GPT"
sys.path.insert(0, str(GARMENT_REC_DIR / "code"))
sys.path.insert(0, str(GARMENT_REC_DIR))
sys.path.insert(0, str(GARMENT_GPT_DIR))

SAM2_CKPT = WEIGHTS_DIR / "sam2" / "sam2_hiera_large.pt"

# ---- EC2 proxy callback URL ----
EC2_CALLBACK_URL = os.getenv("EC2_CALLBACK_URL", "https://korra.work/api/v2/garment/callback")

# ---- globals (populated by load_models) ----
rembg_remove = None
predict_fn = None
sam2_model = None
garmentrec_model = None
garmentgpt_predictor = None

# ---- async model loading state ----
models_loaded = threading.Event()
loading_state = {}
loading_errors = {}

# ---- async job queue ----
jobs = {}

# ---- health tracking ----
SERVER_START_TIME = time.time()
REQUEST_COUNT = 0
ERROR_COUNT = 0
MODEL_LOAD_TIMES = {}

# ---- structured error codes (Phase 31) ----
ERROR_CODES = {
    "CUDA_OOM": {"code": "CUDA_OOM", "message": "GPU out of memory", "recoverable": True},
    "SAM2_LOAD_FAILED": {"code": "SAM2_LOAD_FAILED", "message": "SAM2 model failed to load", "recoverable": True},
    "GARMENTREC_LOAD_FAILED": {"code": "GARMENTREC_LOAD_FAILED", "message": "GarmentRec model failed to load", "recoverable": True},
    "GARMENTGPT_LOAD_FAILED": {"code": "GARMENTGPT_LOAD_FAILED", "message": "GarmentGPT model failed to load", "recoverable": True},
    "REMBG_FAILED": {"code": "REMBG_FAILED", "message": "Background removal failed", "recoverable": True},
    "MODEL_CORRUPT": {"code": "MODEL_CORRUPT", "message": "Model checkpoint is corrupt or incompatible", "recoverable": False},
    "INFERENCE_FAILED": {"code": "INFERENCE_FAILED", "message": "Model inference failed", "recoverable": True},
    "TIMEOUT": {"code": "TIMEOUT", "message": "Request timed out", "recoverable": True},
    "INVALID_IMAGE": {"code": "INVALID_IMAGE", "message": "Invalid or unreadable image", "recoverable": False},
    "IMAGE_TOO_LARGE": {"code": "IMAGE_TOO_LARGE", "message": "Image exceeds size limit", "recoverable": False},
    "UNKNOWN": {"code": "UNKNOWN", "message": "An unexpected error occurred", "recoverable": False},
}


def _structured_error(err_key, detail=None, extra=None):
    """Build a structured error response dict."""
    entry = ERROR_CODES.get(err_key, ERROR_CODES["UNKNOWN"])
    out = {"error": dict(entry)}
    if detail:
        out["error"]["detail"] = str(detail)
    if extra:
        out["error"]["extra"] = extra
    return out


def _write_error_context(error_key, detail=None, request_path=None):
    """Write rich error context to /kaggle/working/last_error.txt (Phase 33)."""
    try:
        import datetime
        parts = [
            f"=== Error at {datetime.datetime.utcnow().isoformat()}Z ===",
            f"Code: {error_key}",
            f"Detail: {detail}" if detail else "",
            f"Path: {request_path}" if request_path else "",
            f"GPU allocated: {torch.cuda.memory_allocated() / (1024**3):.2f}GB" if torch.cuda.is_available() else "GPU: N/A",
            f"GPU reserved: {torch.cuda.memory_reserved() / (1024**3):.2f}GB" if torch.cuda.is_available() else "",
            f"Models: rembg={rembg_remove is not None} sam2={sam2_model is not None} " +
            f"garmentrec={garmentrec_model is not None} garmentgpt={garmentgpt_predictor is not None}",
            f"Loading state: {loading_state}",
            "=" * 60,
        ]
        open("/kaggle/working/last_error.txt", "w").write("\n".join(parts))
    except Exception:
        pass


# ---- retry logic (Phase 32) ----
def _retry_on_oom(func, max_retries=2):
    """Retry a model load if it fails with CUDA OOM, clearing cache between attempts."""
    for attempt in range(max_retries + 1):
        try:
            return func()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and attempt < max_retries:
                logger.warning(f"CUDA OOM on attempt {attempt+1}/{max_retries+1}, clearing cache and retrying...")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise


# ═══════════════════════════════════════════════════════════════
#  SAM2 segmentation (with checkpoint compatibility patch)
# ═══════════════════════════════════════════════════════════════

def _load_sam2():
    """Load SAM2 on CPU. Handles checkpoint version mismatches by filtering unexpected keys.
    Disables CUDA cache warmup to prevent OOM on restart (leaked GPU memory from previous process)."""
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    ckpt = str(SAM2_CKPT)
    if not Path(ckpt).exists():
        raise FileNotFoundError(f"SAM2 checkpoint not found at {ckpt}")

    # Patch build_sam2 to filter mismatched keys (newer sam2 package vs older checkpoint)
    import sam2.build_sam as _build_mod
    _orig_load_checkpoint = _build_mod._load_checkpoint

    def _patched_load_checkpoint(model, ckpt_path):
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if "model" in sd:
            sd = sd["model"]
        model_keys = set(model.state_dict().keys())
        ckpt_keys = set(sd.keys())
        unexpected = ckpt_keys - model_keys
        if unexpected:
            logger.warning(f"SAM2: filtering {len(unexpected)} unexpected keys from checkpoint: {unexpected}")
        filtered = {k: v for k, v in sd.items() if k in model_keys}
        model.load_state_dict(filtered, strict=False)

    _build_mod._load_checkpoint = _patched_load_checkpoint
    try:
        # Disable SAM2's CUDA cache warmup to prevent OOM on server restart.
        # The warmup hardcodes torch.device("cuda") in PositionEmbeddingSine.__init__,
        # bypassing the torch.device("cpu") context. Hydra overrides disable it.
        hydra_overrides = [
            "++model.image_encoder.neck.position_encoding.warmup_cache=false",
            "++model.memory_encoder.position_encoding.warmup_cache=false",
        ]
        with torch.device("cpu"):
            m = build_sam2("sam2_hiera_l.yaml", ckpt, device=CPU_DEVICE,
                           hydra_overrides_extra=hydra_overrides)
    finally:
        _build_mod._load_checkpoint = _orig_load_checkpoint

    pred = SAM2ImagePredictor(m)
    torch.cuda.empty_cache()
    gc.collect()
    logger.info("SAM2 loaded on CPU")
    return m, pred


def segment_garment(image: Image.Image) -> np.ndarray:
    """Real SAM2-based garment segmentation. Returns a binary mask (H,W) uint8 0/255."""
    global predict_fn
    img_np = np.array(image.convert("RGB"))
    h, w = img_np.shape[:2]
    predict_fn.set_image(img_np)
    masks, scores, _ = predict_fn.predict(
        point_coords=np.array([[w // 2, h // 2]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    best = masks[scores.argmax()]
    return (best.astype(np.uint8) * 255)


# ═══════════════════════════════════════════════════════════════
#  GarmentRec — 3D mesh from a single image
# ═══════════════════════════════════════════════════════════════

def _load_garmentrec():
    """Load GarmentRec model on CPU. Fails if assets or SMPL model are missing."""
    import pymeshlab as _pml
    _Percentage = getattr(_pml, 'PercentageValue', None) or getattr(_pml, 'Percentage', None)
    if _Percentage is not None:
        _pml.Percentage = _Percentage
    from module.ImageReconstructModel import ImageReconstructModel
    from module.SkinWeightModel import SkinWeightNet
    import pickle

    gar_dir = GARMENT_REC_DIR / "code"
    smpl_path = GARMENT_REC_DIR / "smpl_pytorch" / "model" / "neutral_smpl_with_cocoplus_reg.txt"
    if not smpl_path.exists():
        raise FileNotFoundError(
            f"SMPL model not found at {smpl_path}. "
            "Download from https://smpl.is.tue.mpg.de/ and place the file there."
        )
    midpair_path = gar_dir.parent / "data" / "midpairs.pkl"
    dense_midpair_path = gar_dir.parent / "data" / "dense_midpairs.pkl"
    pca_folder = gar_dir.parent / "data" / "tmps"
    dense_template_folder = gar_dir.parent / "data"
    model_path = gar_dir.parent / "models" / "mrf_0.1_shading_0.1" / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"

    for p in [midpair_path, dense_midpair_path, pca_folder, model_path]:
        if not p.exists():
            raise FileNotFoundError(f"GarmentRec asset not found: {p}")

    with open(midpair_path, "rb") as f:
        midpairs = pickle.load(f)
    with open(dense_midpair_path, "rb") as f:
        dense_midpairs = pickle.load(f)

    garments = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt", "Shorts", "Pants"]
    garmentvnums = [1954, 1954, 2468, 2468, 678, 1180]
    upper_type_num = 4
    pca_dim = 64
    tran_mean = [0.0, 0.0, 0.0]

    skin_net = SkinWeightNet(4, True)
    device = CPU_DEVICE

    net = ImageReconstructModel(
        skin_net,
        with_classification=True,
        tran_mean=tran_mean,
        garments=garments,
        garmentvnums=garmentvnums,
        upper_type_num=upper_type_num,
        pca_folder=str(pca_folder),
        pca_dim=pca_dim,
        smpl_model_path=str(smpl_path),
        midpairs=midpairs,
        infer_camera=True,
        infer_tex=True,
        inferring=True,
        use_detail=True,
        mesh_save_folder=None,
        vis_save_folder=None,
        dense_template_folder=str(dense_template_folder),
        displacement_scale=0.005,
        upsample_dismap=False,
        use_neighbor=True,
        device=device,
    )
    try:
        state = torch.load(str(model_path), map_location=device, weights_only=False)
    except (EOFError, pickle.UnpicklingError, RuntimeError) as _e:
        logger.error(f"Corrupt GarmentRec weights at {model_path}: {_e}")
        logger.error("Attempting re-download via hf_hub_download...")
        from huggingface_hub import hf_hub_download
        model_path.unlink(missing_ok=True)
        try:
            _new = hf_hub_download(
                repo_id='jacobthankgod4/smpl-model-garmentrec',
                filename='mrf_0.1_shading_0.1_pca64_ep100_bth0.pth',
                local_dir=str(model_path.parent),
                local_dir_use_symlinks=False,
            )
            state = torch.load(str(_new), map_location=device, weights_only=False)
            logger.info(f"Re-downloaded OK ({Path(_new).stat().st_size/(1024*1024):.0f} MB)")
        except Exception as _e2:
            raise RuntimeError(
                f"GarmentRec weights corrupt and re-download failed: {_e2}"
            ) from _e
    # Fix checkpoint mismatches: transpose SMPL matrices, skip incompatible keys
    model_state = net.state_dict()
    fixed_state = {}
    skipped = []
    for key in state:
        if key not in model_state:
            skipped.append(key)
            continue
        ckpt_shape = state[key].shape
        model_shape = model_state[key].shape
        if ckpt_shape == model_shape:
            fixed_state[key] = state[key]
        elif len(ckpt_shape) == 2 and len(model_shape) == 2 and ckpt_shape[0] == model_shape[1] and ckpt_shape[1] == model_shape[0]:
            logger.warning("Transposing %s: %s -> %s", key, ckpt_shape, model_shape)
            fixed_state[key] = state[key].T.contiguous()
        else:
            logger.warning("Skipping %s: ckpt %s != model %s", key, ckpt_shape, model_shape)
            skipped.append(key)
    if skipped:
        logger.info("Skipped %d mismatched keys in state_dict", len(skipped))
    net.load_state_dict(fixed_state, strict=False)

    # Fix SMPL J_regressor orientation: standard SMPL stores (24, 6890) but
    # GarmentRec's SMPL.py expects (6890, 24) for matmul(v_shaped, J_regressor).
    if hasattr(net.smpl, 'J_regressor') and net.smpl.J_regressor.shape == (24, 6890):
        net.smpl.J_regressor.data = net.smpl.J_regressor.T.contiguous()
        logger.info("Patched SMPL J_regressor: transposed (24,6890) -> (6890,24)")

    net = net.to(device)
    net.eval()
    logger.info("GarmentRec loaded on CPU")
    return net


def run_garmentrec(net, image: Image.Image, temp_dir: str) -> dict:
    """Run GarmentRec inference, save mesh to temp_dir, return mesh data."""
    import cv2
    import trimesh

    net.mesh_save_folder = temp_dir

    img_np = np.array(image.convert("RGB"))
    img_np = cv2.resize(img_np, (540, 540))
    img_np = img_np.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(CPU_DEVICE)
    names = np.array(["input.png"])

    cam_k = torch.Tensor(
        [[3.0375e03, 0.0, 270.0], [0.0, 3.0375e03, 270.0], [0.0, 0.0, 1.0]]
    ).to(CPU_DEVICE)
    bcnet_tran_mean = [-0.010962, 0.28778, 12.973]
    tran_mean = [0.0, 0.0, 0.0]
    cam_Rt = torch.tensor(
        [
            [1, 0, 0, tran_mean[0] - bcnet_tran_mean[0]],
            [0, 1, 0, tran_mean[1] - bcnet_tran_mean[1]],
            [0, 0, 1, tran_mean[2] - bcnet_tran_mean[2]],
        ]
    )
    cam_k = cam_k.matmul(cam_Rt).to(CPU_DEVICE)

    imgs_perg = torch.cat((img_tensor, img_tensor), 1).reshape(-1, 3, 540, 540)
    input_gtypes = np.array([[-1, -1]])

    with torch.no_grad():
        up_prob, bottom_prob, cam_Rs, cam_Ts, dis_maps = net(
            img_tensor, names, gtypes=input_gtypes, cam_k=cam_k, imgs_perg=imgs_perg
        )

    up_idx = up_prob.argmax(dim=1).item()
    bottom_idx = bottom_prob.argmax(dim=1).item()
    up_name = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt"][up_idx]
    bi = bottom_idx - 4 if bottom_idx >= 4 else bottom_idx
    bottom_name = ["Shorts", "Pants"][bi]
    logger.info(f"GarmentRec classified: upper={up_name}, lower={bottom_name}")

    up_path = os.path.join(temp_dir, "input_up.obj")
    bottom_path = os.path.join(temp_dir, "input_bottom.obj")

    mesh_data = {"upper": None, "lower": None}
    if os.path.exists(up_path):
        mesh = trimesh.load(up_path)
        v = mesh.vertices
        f = mesh.faces
        mesh_data["upper"] = {
            "vertices": v.tolist(),
            "faces": f.tolist(),
            "type": up_name,
        }
    if os.path.exists(bottom_path):
        mesh = trimesh.load(bottom_path)
        v = mesh.vertices
        f = mesh.faces
        mesh_data["lower"] = {
            "vertices": v.tolist(),
            "faces": f.tolist(),
            "type": bottom_name,
        }
    return mesh_data


# ═══════════════════════════════════════════════════════════════
#  GarmentGPT — sewing pattern from a single image
# ═══════════════════════════════════════════════════════════════

def _load_garmentgpt():
    """Load GarmentGPT predictor. LLM on GPU, codecs on CPU."""
    from main import GarmentPredictor

    gpt_dir = GARMENT_GPT_DIR
    llm_path = str(gpt_dir / "checkpoints" / "vlm" / "checkpoint-12844")
    codec_cfg = str(gpt_dir / "configs" / "config_vq1024_resres_aug_decay0.99_q5_gcd_nl8_ld512.yaml")
    rt_cfg = str(gpt_dir / "configs" / "config_rt_euler.yaml")

    for p in [llm_path, codec_cfg, rt_cfg]:
        if not Path(p).exists():
            raise FileNotFoundError(f"GarmentGPT asset not found: {p}")

    _orig_cwd = os.getcwd()
    os.chdir(str(gpt_dir))
    try:
        predictor = GarmentPredictor(
            llm_model_path=llm_path,
            codec_config_path=codec_cfg,
            rt_config_path=rt_cfg,
            device=GPU_DEVICE,
        )
    finally:
        os.chdir(_orig_cwd)

    predictor.codec_model = predictor.codec_model.to(CPU_DEVICE)
    predictor.rt_model = predictor.rt_model.to(CPU_DEVICE)
    torch.cuda.empty_cache()
    logger.info("GarmentGPT loaded (LLM on GPU, codecs on CPU)")
    return predictor


def run_garmentgpt(predictor, image: Image.Image) -> dict:
    """Save image to a temp file, run GarmentGPT predict, return GCD JSON."""
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        tmp_path = f.name
        image.save(tmp_path, "PNG")
    try:
        result = predictor.predict(image_path=tmp_path)
        if result is None:
            raise RuntimeError("GarmentGPT prediction returned None")
        return result
    finally:
        try:
            os.unlink(tmp_path)
        except Exception:
            pass


# ═══════════════════════════════════════════════════════════════
#  rembg (background removal) — with PIL fallback
# ═══════════════════════════════════════════════════════════════

def _simple_remove_background(image, threshold=240):
    """PIL-based background removal fallback when rembg is unavailable."""
    import numpy as np
    arr = np.array(image.convert("RGBA"), dtype=np.uint8)
    bg = np.all(arr[:,:,:3] > threshold, axis=2)
    arr[bg, 3] = 0
    result = Image.fromarray(arr, mode="RGBA")
    canvas = Image.new("RGB", result.size, (255, 255, 255))
    canvas.paste(result, mask=result.split()[3])
    return canvas


def _load_rembg():
    """Robust rembg loader with PIL fallback. Handles numpy 2.x / scipy incompat."""
    global rembg_remove
    import subprocess, sys as _sys

    # Step 1: Ensure onnxruntime
    try:
        import onnxruntime
    except ImportError:
        logger.warning("onnxruntime not found, installing...")
        try:
            subprocess.check_call(
                [_sys.executable, "-m", "pip", "install", "onnxruntime"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            logger.warning("onnxruntime install failed, using PIL fallback")
            rembg_remove = _simple_remove_background
            return

    # Step 2: Upgrade scipy for numpy 2.x compat (rembg pulls scipy via pymatting)
    try:
        import scipy, packaging.version as _v
        if _v.Version(scipy.__version__) < _v.Version("1.14.0"):
            raise ImportError
    except (ImportError, ValueError, AttributeError):
        logger.warning("scipy too old for numpy 2.x, upgrading...")
        try:
            subprocess.check_call(
                [_sys.executable, "-m", "pip", "install", "--upgrade", "scipy>=1.14.1"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            pass

    # Step 3: Import rembg (retry with reinstall on failure)
    for attempt in range(2):
        try:
            from rembg import remove
            break
        except BaseException:
            if attempt == 0:
                logger.warning("rembg import failed, installing rembg[cpu]...")
                try:
                    subprocess.check_call(
                        [_sys.executable, "-m", "pip", "install", "rembg[cpu]"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                    )
                except subprocess.CalledProcessError:
                    pass
            else:
                logger.warning("rembg import failed after reinstall, using PIL fallback")
                rembg_remove = _simple_remove_background
                return

    # Step 4: Warm up (downloads u2net ~176MB)
    try:
        dummy = Image.new("RGB", (8, 8), (255, 255, 255))
        remove(dummy)
        rembg_remove = remove
        logger.info("rembg + u2net ready")
    except Exception:
        logger.warning("rembg warmup failed, using PIL fallback")
        rembg_remove = _simple_remove_background


# ═══════════════════════════════════════════════════════════════
#  Model loading
# ═══════════════════════════════════════════════════════════════

def _log_gpu_memory(tag: str = ""):
    """Log current GPU memory state for debugging CUDA OOM issues."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        logger.info(f"[GPU MEM] {tag} allocated={allocated:.2f}GB reserved={reserved:.2f}GB")


def _loading_thread():
    """Run model loading in a background thread so uvicorn can start immediately."""
    global rembg_remove, predict_fn, sam2_model, garmentrec_model, garmentgpt_predictor
    global loading_state, loading_errors, MODEL_LOAD_TIMES
    logger.info("Background model loading thread started")
    try:
        t0 = time.time()
        loading_state["rembg"] = "loading"
        _load_rembg()
        loading_state["rembg"] = "loaded"
        MODEL_LOAD_TIMES["rembg"] = time.time() - t0
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        t1 = time.time()
        loading_state["sam2"] = "loading"
        _log_gpu_memory("before_sam2")
        sam2_model, predict_fn = _retry_on_oom(lambda: _load_sam2(), max_retries=2)
        loading_state["sam2"] = "loaded"
        MODEL_LOAD_TIMES["sam2"] = time.time() - t1
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        t2 = time.time()
        loading_state["garmentrec"] = "loading"
        _log_gpu_memory("before_garmentrec")
        garmentrec_model = _load_garmentrec()
        loading_state["garmentrec"] = "loaded"
        MODEL_LOAD_TIMES["garmentrec"] = time.time() - t2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        t3 = time.time()
        loading_state["garmentgpt"] = "loading"
        _log_gpu_memory("before_garmentgpt")
        garmentgpt_predictor = _load_garmentgpt()
        loading_state["garmentgpt"] = "loaded"
        MODEL_LOAD_TIMES["garmentgpt"] = time.time() - t3
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        _log_gpu_memory("all_loaded")
        logger.info("All models loaded successfully")
    except Exception as e:
        logger.error(f"Model loading thread failed: {e}")
        loading_errors["fatal"] = traceback.format_exc()
    finally:
        models_loaded.set()


# ═══════════════════════════════════════════════════════════════
#  Helper: build ZIP from pipeline results
# ═══════════════════════════════════════════════════════════════

def build_zip(image_id: str, mesh_data: dict = None, pattern_data: dict = None) -> bytes:
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        if mesh_data:
            for part in ("upper", "lower"):
                md = mesh_data.get(part)
                if md:
                    obj_lines = [f"# Garment {part} ({md['type']})"]
                    for v in md["vertices"]:
                        obj_lines.append(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}")
                    for fa in md["faces"]:
                        obj_lines.append(f"f {fa[0]+1} {fa[1]+1} {fa[2]+1}")
                    zf.writestr(f"{image_id}/mesh_{part}.obj", "\n".join(obj_lines))
        if pattern_data:
            zf.writestr(f"{image_id}/sewing_pattern.json", json.dumps(pattern_data, indent=2))
        meta = {
            "image_id": image_id,
            "garmentrec": mesh_data is not None,
            "garmentgpt": pattern_data is not None,
            "generated_at": datetime.utcnow().isoformat(),
        }
        zf.writestr(f"{image_id}/metadata.json", json.dumps(meta, indent=2))
    buffer.seek(0)
    return buffer.read()


# ═══════════════════════════════════════════════════════════════
#  Helper: post result to EC2 proxy callback
# ═══════════════════════════════════════════════════════════════

def post_result_to_proxy(job_id: str, result_zip: bytes):
    """POST ZIP to EC2 proxy callback endpoint (fire-and-forget)."""
    try:
        import httpx
        files = {"file": (f"{job_id}.zip", io.BytesIO(result_zip), "application/zip")}
        data = {"job_id": job_id}
        resp = httpx.post(EC2_CALLBACK_URL, files=files, data=data, timeout=30)
        logger.info(f"Posted result to proxy: {resp.status_code}")
    except Exception as e:
        logger.error(f"Failed to post result to proxy: {e}")


# ═══════════════════════════════════════════════════════════════
#  Background task: full pipeline
# ═══════════════════════════════════════════════════════════════

async def process_full_pipeline(job_id: str, image_bytes: bytes, include_mesh: bool, include_pattern: bool):
    """Runs in background. Builds ZIP, stores in jobs dict, posts to EC2 proxy."""
    try:
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        nobg = rembg_remove(image)
        mask = segment_garment(image)

        mesh_data = None
        pattern_data = None

        if include_mesh and garmentrec_model is not None:
            with tempfile.TemporaryDirectory() as tmp:
                mesh_data = run_garmentrec(garmentrec_model, nobg, tmp)
            gc.collect()

        if include_pattern and garmentgpt_predictor is not None:
            pattern_data = run_garmentgpt(garmentgpt_predictor, nobg)

        result_zip = build_zip(job_id, mesh_data, pattern_data)
        jobs[job_id]["status"] = "completed"
        jobs[job_id]["result_zip"] = result_zip

        # Post to EC2 proxy for persistent storage
        post_result_to_proxy(job_id, result_zip)
        logger.info(f"Job {job_id} completed successfully")
    except Exception as e:
        import traceback as tb
        err = tb.format_exc()
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        try:
            open("/kaggle/working/last_error.txt", "w").write(err)
        except Exception:
            pass
        logger.error(f"Job {job_id} failed: {e}")


# ═══════════════════════════════════════════════════════════════
#  FastAPI app
# ═══════════════════════════════════════════════════════════════

@asynccontextmanager
async def lifespan(app):
    # Start model loading in background thread — uvicorn serves immediately
    thread = threading.Thread(target=_loading_thread, daemon=True)
    thread.start()
    yield
    # Graceful shutdown: release models and free GPU memory
    global rembg_remove, predict_fn, sam2_model, garmentrec_model, garmentgpt_predictor
    logger.info("Shutting down — releasing models...")
    rembg_remove = None
    predict_fn = None
    sam2_model = None
    garmentrec_model = None
    garmentgpt_predictor = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    logger.info("Shutdown complete")


app = FastAPI(title="Garment Reconstruction API", lifespan=lifespan)


@app.middleware("http")
async def catch_all_errors(request, call_next):
    global REQUEST_COUNT, ERROR_COUNT
    path = request.url.path
    include_in_count = not path.startswith("/health") and not path.startswith("/ready")
    if include_in_count:
        REQUEST_COUNT += 1
    try:
        return await call_next(request)
    except BaseException as e:
        if include_in_count:
            ERROR_COUNT += 1
        import traceback as tb
        err_type = type(e).__name__
        err_tb = tb.format_exc()
        err_key = "CUDA_OOM" if "out of memory" in str(e).lower() else "UNKNOWN"
        _write_error_context(err_key, detail=err_tb, request_path=path)
        tb.print_exc()
        logger.error(f"UNCAUGHT {err_key}: {err_type}: {e}")
        return JSONResponse(status_code=500, content=_structured_error(err_key, detail=str(e)))


@app.get("/health")
async def health(ready: bool = False, verbose: bool = False):
    global REQUEST_COUNT, ERROR_COUNT
    gpu_allocated = None
    gpu_reserved = None
    gpu_ok = True
    if torch.cuda.is_available():
        gpu_allocated = torch.cuda.memory_allocated() / (1024**3)
        gpu_reserved = torch.cuda.memory_reserved() / (1024**3)
        gpu_ok = gpu_allocated < 15.0  # 15GB threshold

    base = {
        "status": "healthy" if models_loaded.is_set() else "loading",
        "device": GPU_DEVICE,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "models_ready": models_loaded.is_set(),
        "loading_state": loading_state,
        "loading_errors": loading_errors if loading_errors else None,
        "rembg": rembg_remove is not None,
        "sam2": sam2_model is not None,
        "garmentrec": garmentrec_model is not None,
        "garmentgpt": garmentgpt_predictor is not None,
        "jobs_processing": sum(1 for j in jobs.values() if j["status"] == "processing"),
        "uptime_sec": time.time() - SERVER_START_TIME,
        "request_count": REQUEST_COUNT,
        "error_count": ERROR_COUNT,
        "gpu_ok": gpu_ok,
    }
    if verbose or ready:
        gpu_allocated = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
        gpu_reserved = torch.cuda.memory_reserved() / (1024**3) if torch.cuda.is_available() else 0
        base["gpu_memory_gb"] = {
            "allocated": round(gpu_allocated, 2) if gpu_allocated else None,
            "reserved": round(gpu_reserved, 2) if gpu_reserved else None,
            "threshold_15gb_ok": gpu_ok,
        }
        base["model_load_times_sec"] = {k: round(v, 2) for k, v in MODEL_LOAD_TIMES.items()}

    if ready and not models_loaded.is_set():
        return JSONResponse(status_code=503, content=base)

    if not gpu_ok:
        return JSONResponse(status_code=503, content=base)

    return base


@app.get("/ready")
async def ready():
    if models_loaded.is_set():
        return {"status": "ready"}
    return JSONResponse(status_code=503, content={
        "status": "loading",
        "loading_state": loading_state,
        "loading_errors": loading_errors if loading_errors else None,
    })


@app.get("/health/deep")
async def health_deep():
    """Run a tiny inference to verify models actually work (slow check)."""
    if not models_loaded.is_set():
        return JSONResponse(status_code=503, content={"status": "loading", "detail": "Models not ready"})
    try:
        # Test rembg: create 8x8 dummy image
        from PIL import Image
        import numpy as np
        dummy = Image.new("RGB", (8, 8), (255, 255, 255))
        _ = rembg_remove(dummy)
        return {"status": "ready", "checks": {"rembg": "ok"}}
    except Exception as e:
        return JSONResponse(status_code=503, content={"status": "degraded", "checks": {"rembg": f"fail: {e}"}})


# ─────────────────────────────────────────────────────────────
#  Async reconstruct (returns job_id immediately)
# ─────────────────────────────────────────────────────────────

@app.post("/api/v1/reconstruct")
async def reconstruct(
    file: UploadFile = File(...),
    include_mesh: bool = True,
    include_pattern: bool = True,
    user_id: str = "",
):
    image_bytes = await file.read()
    if len(image_bytes) > 10 * 1024 * 1024:
        raise HTTPException(400, detail=_structured_error("IMAGE_TOO_LARGE"))
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail="File must be an image"))
    # Validate image can be opened and has valid dimensions
    try:
        _test_img = Image.open(io.BytesIO(image_bytes))
        _test_img.verify()
        _test_img = Image.open(io.BytesIO(image_bytes))
        if _test_img.width < 32 or _test_img.height < 32:
            raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=f"Image too small ({_test_img.width}x{_test_img.height}), min 32x32"))
        if _test_img.width > 4096 or _test_img.height > 4096:
            raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=f"Image too large ({_test_img.width}x{_test_img.height}), max 4096x4096"))
    except HTTPException:
        raise
    except Exception as _e:
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=str(_e)))
    if rembg_remove is None:
        raise HTTPException(503, detail=_structured_error("REMBG_FAILED", detail="rembg not available"))

    job_id = str(uuid.uuid4())[:8]
    jobs[job_id] = {
        "status": "processing",
        "result_zip": None,
        "error": None,
        "user_id": user_id,
        "created_at": datetime.utcnow().isoformat(),
    }

    # Spawn background task (non-blocking)
    asyncio.create_task(process_full_pipeline(job_id, image_bytes, include_mesh, include_pattern))

    return {"job_id": job_id, "status": "processing"}


@app.get("/api/v1/job/{job_id}")
async def get_job(job_id: str):
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, "Job not found")
    if job["status"] == "completed":
        return StreamingResponse(
            io.BytesIO(job["result_zip"]),
            media_type="application/zip",
            headers={"Content-Disposition": f"attachment; filename=garment_{job_id}.zip"},
        )
    elif job["status"] == "failed":
        return JSONResponse(status_code=500, content={"status": "failed", "error": job["error"]})
    return {"status": "processing"}


# ─────────────────────────────────────────────────────────────
#  Split pipeline endpoints (each < 100s)
# ─────────────────────────────────────────────────────────────

@app.post("/api/v1/segment")
async def segment_endpoint(file: UploadFile = File(...)):
    """Step 1: Background removal + SAM2 segmentation (~20s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    nobg = rembg_remove(image)
    mask = segment_garment(image)
    buf = io.BytesIO()
    Image.fromarray(mask).save(buf, "PNG")
    buf.seek(0)
    elapsed = time.time() - start
    return StreamingResponse(buf, media_type="image/png", headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.post("/api/v1/mesh")
async def mesh_endpoint(file: UploadFile = File(...)):
    """Step 2: GarmentRec 3D mesh (~50s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    if rembg_remove is None:
        raise HTTPException(503, "rembg not available")
    nobg = rembg_remove(image)
    with tempfile.TemporaryDirectory() as tmp:
        mesh_data = run_garmentrec(garmentrec_model, nobg, tmp)
    gc.collect()
    elapsed = time.time() - start
    return JSONResponse(content=mesh_data, headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.post("/api/v1/pattern")
async def pattern_endpoint(file: UploadFile = File(...)):
    """Step 3: GarmentGPT sewing pattern (~50s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    if rembg_remove is None:
        raise HTTPException(503, "rembg not available")
    nobg = rembg_remove(image)
    pattern_data = run_garmentgpt(garmentgpt_predictor, nobg)
    elapsed = time.time() - start
    return JSONResponse(content=pattern_data, headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.get("/debug/error")
async def debug_error():
    try:
        return {"error": open("/kaggle/working/last_error.txt").read()}
    except Exception as e:
        return {"error": f"No error file: {e}"}


if __name__ == "__main__":
    import signal
    import nest_asyncio
    nest_asyncio.apply()

    def _graceful_exit(signum, frame):
        logger.info(f"Received signal {signum}, cleaning up GPU memory...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("GPU memory cleaned, exiting.")
        raise SystemExit(0)

    signal.signal(signal.SIGTERM, _graceful_exit)
    signal.signal(signal.SIGINT, _graceful_exit)
    uvicorn.run(app, host="0.0.0.0", port=8000, limit_concurrency=2, backlog=10)
'''

## Cell 5: Start Server + Tunnel + Keep-Alive

In [ ]:
"""
Cell 5: Start Server + Tunnel + Keep-Alive (fp16 + OOM fix + int8 shim)
Paste this entire cell into Kaggle notebook Cell 5.
"""

# ---- vLLM compatibility layer (SHIM) ----
# Required because Garment-GPT main.py imports vllm, which isn't compatible with Kaggle P100 (sm_60).
import sys, types, torch

def _activate_vllm_shim():
    import importlib
    _orig_find_spec = importlib.util.find_spec
    def _patched_find_spec(name, *args, **kwargs):
        if name == 'vllm':
            return None
        return _orig_find_spec(name, *args, **kwargs)
    importlib.util.find_spec = _patched_find_spec

    class _HF_LLM:
        def __init__(self, model, **kwargs):
            from transformers import AutoProcessor, AutoConfig
            import torch

            use_8bit = False
            gpu_label = "cpu"
            if torch.cuda.is_available():
                cap = torch.cuda.get_device_capability()
                gpu_label = f"sm_{cap[0]}{cap[1]}"
                if cap >= (7, 0):
                    use_8bit = True

            print(f'[vllm-shim] Loading model {model} on {gpu_label}...')
            self._processor = AutoProcessor.from_pretrained(model, trust_remote_code=True)
            cfg_kwargs = dict(kwargs)
            cfg_kwargs.setdefault('trust_remote_code', True)
            cfg = AutoConfig.from_pretrained(model, **cfg_kwargs)
            if hasattr(cfg, 'vision_config'):
                try:
                    from transformers import AutoModelForVision2Seq
                    model_class = AutoModelForVision2Seq
                except ImportError:
                    from transformers import LlavaForConditionalGeneration
                    model_class = LlavaForConditionalGeneration
            else:
                from transformers import AutoModelForCausalLM
                model_class = AutoModelForCausalLM

            load_kwargs = dict(
                device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
            )
            if use_8bit:
                from transformers import BitsAndBytesConfig
                load_kwargs['quantization_config'] = BitsAndBytesConfig(
                    load_in_8bit=True, llm_int8_threshold=6.0,
                )

            self._model = model_class.from_pretrained(model, **load_kwargs)
            self._model.eval()
            print(f'[vllm-shim] Model loaded on {self._model.device}')

        def generate(self, inputs_list, sampling_params):
            results = []
            for inp in inputs_list:
                prompt = inp.get('prompt', '')
                images = inp.get('multi_modal_data', {}).get('image', [])
                if images:
                    proc = self._processor(text=prompt, images=images[0], return_tensors='pt').to(self._model.device)
                else:
                    proc = self._processor(text=prompt, return_tensors='pt').to(self._model.device)
                mt = getattr(sampling_params, 'max_tokens', 4096)
                tp = getattr(sampling_params, 'temperature', 0.1)
                with torch.no_grad():
                    out = self._model.generate(**proc, max_new_tokens=min(mt, 4096),
                        do_sample=tp > 0, temperature=max(tp, 0.01), top_p=0.9)
                gen = self._processor.decode(out[0], skip_special_tokens=False)
                class _O:
                    def __init__(self, text): self.text = text
                class _RO:
                    def __init__(self, outputs): self.outputs = outputs
                results.append(_RO(outputs=[_O(text=gen)]))
            return results

    class _HF_SamplingParams:
        def __init__(self, temperature=0.1, max_tokens=4096, skip_special_tokens=False, seed=42, stop_token_ids=None):
            self.temperature = temperature
            self.max_tokens = max_tokens
            self.skip_special_tokens = skip_special_tokens
            self.seed = seed
            self.stop_token_ids = stop_token_ids or []

    vllm_mod = types.ModuleType('vllm')
    vllm_mod.LLM = _HF_LLM
    vllm_mod.SamplingParams = _HF_SamplingParams
    sys.modules['vllm'] = vllm_mod
    print('[vllm-shim] vLLM compatibility shim installed (transformers + int8 backend)')

_activate_vllm_shim()

import os, subprocess, time, re, requests, shutil, threading
def register_tunnel(url):
    """POST tunnel URL to EC2 proxy + write output file for rotation script."""
    if not url or not url.startswith("http"):
        print(f"[Tunnel] Skipping registration: invalid URL '{url}'")
        return
    from pathlib import Path
    out = Path("/kaggle/working/output")
    out.mkdir(exist_ok=True)
    out.joinpath("tunnel_url.txt").write_text(url)
    print(f"[Tunnel] Written to {out/'tunnel_url.txt'}")
    for attempt in range(5):
        try:
            import requests as _req
            r = _req.post("https://korra.work/api/v2/garment/internal/tunnel",
                          json={"url": url}, timeout=10)
            body = r.text[:500]
            if r.status_code == 200:
                print(f"[Tunnel] Registered with EC2 proxy (attempt {attempt+1}): {body[:200]}")
                return
            print(f"[Tunnel] EC2 proxy HTTP {r.status_code} (attempt {attempt+1}): {body[:200]}")
        except Exception as e:
            print(f"[Tunnel] EC2 proxy error (attempt {attempt+1}): {e}")
        time.sleep(5)


# Locate cloudflared binary
CLOUDFLARED = shutil.which("cloudflared") or "/kaggle/working/cloudflared"
if not os.path.exists(CLOUDFLARED):
    raise FileNotFoundError(f"cloudflared not found at {CLOUDFLARED}")

# Kill any old processes
os.system("pkill -f api_server.py 2>/dev/null || true")
os.system("pkill -f cloudflared 2>/dev/null || true")
os.system("fuser -k 8000/tcp 2>/dev/null || true")
time.sleep(2)

tunnel_url = None
_tunnel_start_time = None
_tunnel_restarts = 0
_server_restarts = 0
_keep_alive_backoff = 60  # initial check interval in seconds
_max_server_restarts = 10  # after this, enter degraded mode
_degraded_mode = False
_watchdog_restarts = 0
_max_watchdog_restarts = 5

def _tunnel_reader(stream):
    """Read tunnel output, print it, and watch for tunnel URL."""
    global tunnel_url, _tunnel_start_time, _tunnel_restarts
    try:
        for line in iter(stream.readline, ''):
            if line:
                line = line.rstrip("\n")
                print(f"[TUN] {line}")
                if not tunnel_url:
                    m = re.search(r'https?://[^\s]*trycloudflare\.com[^\s]*', line)
                    if m:
                        tunnel_url = m.group(0)
                        _tunnel_start_time = time.time()
                        _tunnel_restarts += 1
                        print(f"\n{'='*60}\nTUNNEL URL: {tunnel_url}\n{'='*60}")
    except ValueError:
        pass
    finally:
        try:
            stream.close()
        except Exception:
            pass


def start_tunnel():
    global tunnel_url
    print(f"[TUN] Starting cloudflared from {CLOUDFLARED}...")
    p = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    # Drain tunnel logs in background thread (prevents pipe buffer blocking)
    t = threading.Thread(target=_tunnel_reader, args=(p.stdout,), daemon=True)
    t.start()
    # Wait for tunnel URL (up to 90s)
    deadline = time.time() + 90
    while time.time() < deadline:
        if tunnel_url:
            break
        if p.poll() is not None:
            print(f"[TUN] cloudflared died (code {p.returncode}), restarting...")
            return start_tunnel()
        time.sleep(2)
    return p


def _pipe_drainer(stream, prefix, log_file=None):
    """Read from a subprocess pipe continuously and print to stdout."""
    try:
        for line in iter(stream.readline, ''):
            if line:
                print(f"{prefix} {line.strip()}")
                if log_file:
                    log_file.write(line)
                    log_file.flush()
    except ValueError:
        pass  # closed stream
    finally:
        stream.close()
        if log_file:
            log_file.close()


def start_server():
    log_file = open("/kaggle/working/server_logs.txt", "a", buffering=1)
    p = subprocess.Popen(
        ["python", "/kaggle/working/api_server.py"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    t = threading.Thread(target=_pipe_drainer, args=(p.stdout, "[API]", log_file), daemon=True)
    t.start()
    # Wait for server to be ready (up to 60s)
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            r = requests.get("http://localhost:8000/health", timeout=3)
            if r.status_code < 500:
                print("[API] Server ready")
                break
        except Exception:
            time.sleep(1)
            continue
    return p


def kill_server(proc):
    """Gracefully kill server: SIGTERM -> wait 5s -> SIGKILL. Frees CUDA memory."""
    if proc is None:
        return
    pid = proc.pid
    if pid is None:
        return
    try:
        os.kill(pid, 0)  # Check if process exists
    except OSError:
        return  # Already dead
    print(f"[KEEP] Sending SIGTERM to server (PID {pid})...")
    try:
        proc.terminate()  # SIGTERM
        proc.wait(timeout=8)
        print(f"[KEEP] Server PID {pid} terminated gracefully")
    except subprocess.TimeoutExpired:
        print(f"[KEEP] Server PID {pid} didn't exit in 8s, sending SIGKILL...")
        proc.kill()  # SIGKILL
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            print(f"[KEEP] Server PID {pid} still alive after SIGKILL, moving on")
    except Exception as e:
        print(f"[KEEP] Error killing server PID {pid}: {e}")
        try:
            proc.kill()
        except Exception:
            pass
    # Force CUDA memory cleanup after kill
    import gc
    gc.collect()

# Start tunnel first (blocking until URL), then server
t_proc = start_tunnel()
s_proc = start_server()

# Wait then health check
time.sleep(3)
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Health: {r.json()}")
except Exception as e:
    print(f"Health check: {e}")

# Wait up to 90s for tunnel URL
for _wait in range(30):
    if tunnel_url:
        break
    time.sleep(3)
    print(f"[Tunnel] Waiting for URL... ({_wait*3+3}s)")
    if t_proc.poll() is not None:
        print(f"[Tunnel] cloudflared died (code {t_proc.returncode}), restarting...")
        t_proc = start_tunnel()

if tunnel_url:
    print(f"\nReady! Tunnel URL: {tunnel_url}")
    register_tunnel(tunnel_url)
else:
    print(f"\nTunnel URL not yet available — will retry in keep-alive loop")

# === STARTUP TIMEOUT (Phase 44) ===
_STARTUP_TIMEOUT = 120  # max seconds to wait for server + tunnel
_startup_deadline = time.time() + _STARTUP_TIMEOUT
while time.time() < _startup_deadline:
    # Check server health
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        j = r.json()
        server_up = True
        print(f"Health: {j}")
    except Exception as e:
        server_up = False
        print(f"Startup health check: {e}")

    if tunnel_url and server_up:
        print(f"\nReady! Tunnel URL: {tunnel_url}")
        register_tunnel(tunnel_url)
        break

    # Check tunnel
    if t_proc.poll() is not None:
        print(f"[Tunnel] cloudflared died (code {t_proc.returncode}), restarting...")
        t_proc = start_tunnel()

    # Check server
    if s_proc.poll() is not None:
        print(f"[Server] api_server died (code {s_proc.returncode}), restarting...")
        s_proc = start_server()

    time.sleep(5)

if not tunnel_url:
    print(f"\nTunnel URL not yet available — will retry in keep-alive loop")
if not server_up:
    print(f"\nServer not responding — will retry in keep-alive loop")

# === KEEP-ALIVE LOOP (Phases 41-45) ===
print(f"\nKeep-alive running. Initial interval: {_keep_alive_backoff}s.")
import datetime as dt
_health_fails = 0
_tunnel_fails = 0

# === PHASE 45: WATCHDOG-MANAGED KEEP-ALIVE LOOP ===
def _keep_alive_loop():
    """Core keep-alive loop. Runs forever unless an exception escapes."""
    global _health_fails, _tunnel_fails, _server_restarts, _degraded_mode
    global _keep_alive_backoff, t_proc, s_proc, tunnel_url, _tunnel_start_time
    _health_fails = 0
    _tunnel_fails = 0
    while True:
        try:
            # Local health check
            r = requests.get("http://localhost:8000/health", timeout=10)
            j = r.json()
            _health_fails = 0
            # Phase 41: reset backoff on success
            _keep_alive_backoff = max(60, _keep_alive_backoff * 0.75)

            # Update tunnel_url from health if needed
            if tunnel_url and not tunnel_url.startswith('http'):
                tunnel_url = j.get('tunnel_url')

            # Phase 43: Tunnel health check via Cloudflare
            tunnel_alive_via_cloudflare = False
            if tunnel_url:
                try:
                    hr = requests.get(f"{tunnel_url}/health", timeout=10)
                    tunnel_alive_via_cloudflare = hr.status_code == 200
                except Exception:
                    pass

            tunnel_uptime = time.time() - _tunnel_start_time
            now = dt.datetime.now().strftime("%H:%M:%S")
            tu_label = (tunnel_url or 'WAITING')[-55:]
            alive_flag = 'CF' if tunnel_alive_via_cloudflare else 'LOCAL'
            gpu_ok = j.get('gpu_ok', True)
            mode_flag = 'DEGRADED' if _degraded_mode else 'ACTIVE'
            print(f"[{now}] {mode_flag} | GPU: {j.get('gpu','?')} ok={gpu_ok} | Tunnel: {tu_label} [{alive_flag}] up={tunnel_uptime/60:.0f}m | tunnel_restarts={_tunnel_restarts} server_restarts={_server_restarts}")

            if tunnel_url and tunnel_alive_via_cloudflare:
                register_tunnel(tunnel_url)
                _tunnel_fails = 0
            elif tunnel_url and not tunnel_alive_via_cloudflare:
                _tunnel_fails += 1
                print(f"[KEEP] Tunnel URL set but NOT reachable via Cloudflare ({_tunnel_fails}/3). Checking cloudflared process...")
                if t_proc.poll() is not None:
                    print(f"[KEEP] cloudflared died (code {t_proc.returncode}), restarting...")
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                elif _tunnel_fails >= 3:
                    print(f"[KEEP] 3 consecutive tunnel fails — killing cloudflared for fresh tunnel")
                    t_proc.kill()
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                    _tunnel_fails = 0
            elif t_proc.poll() is not None:
                print(f"[KEEP] cloudflared died (code {t_proc.returncode}), restarting...")
                t_proc = start_tunnel()
                register_tunnel(tunnel_url)
                _tunnel_start_time = time.time()
            else:
                _tunnel_fails += 1
                if _tunnel_fails >= 6:
                    print(f"[KEEP] Tunnel still waiting after ~30 min, restarting...")
                    t_proc.kill()
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                    _tunnel_fails = 0

        except Exception as e:
            now = dt.datetime.now().strftime("%H:%M:%S")
            _health_fails += 1
            # Phase 41: exponential backoff on failure
            _keep_alive_backoff = min(600, _keep_alive_backoff * 2.0)

            if _degraded_mode:
                print(f"[{now}] DEGRADED MODE: health fail ({_health_fails}/3), server restart SKIPPED (max restarts {_max_server_restarts} reached)")
                continue

            if _health_fails >= 3:
                _server_restarts += 1
                # Phase 42: max restart limit
                if _server_restarts > _max_server_restarts:
                    _degraded_mode = True
                    print(f"[{now}] Entering DEGRADED MODE after {_server_restarts} server restarts. Tunnel will stay up for manual recovery.")
                    continue

                print(f"[{now}] Health fail {_health_fails}/3 — restarting server ({_server_restarts}/{_max_server_restarts})")
                kill_server(s_proc)
                s_proc = start_server()
                _health_fails = 0
            else:
                print(f"[{now}] Health fail ({_health_fails}/3): {type(e).__name__}: {e} — not restarting yet")

        # Phase 41: adaptive sleep interval
        time.sleep(int(_keep_alive_backoff))


def _watchdog_main():
    """Watchdog wrapper: restarts _keep_alive_loop if it crashes."""
    global _watchdog_restarts, _keep_alive_backoff, _degraded_mode
    while True:
        try:
            _keep_alive_loop()
        except Exception as e:
            _watchdog_restarts += 1
            if _watchdog_restarts > _max_watchdog_restarts:
                print(f"[WATCHDOG] Max restarts ({_max_watchdog_restarts}) reached. Giving up.")
                break
            _keep_alive_backoff = min(600, _keep_alive_backoff * 2.0)
            wait = int(_keep_alive_backoff)
            print(f"[WATCHDOG] Keep-alive crashed ({type(e).__name__}: {e}). Restarting in {wait}s... (attempt {_watchdog_restarts}/{_max_watchdog_restarts})")
            time.sleep(wait)
            _degraded_mode = False
            # Kill and restart tunnel + server from scratch
            if t_proc: t_proc.kill()
            if s_proc: kill_server(s_proc)
            time.sleep(3)
            t_proc = start_tunnel()
            s_proc = start_server()


_watchdog_main()


## Cell 6: Evaluate on Zenodo Dataset (optional)
Run after server is up. Downloads test set (~4GB), evaluates Chamfer distance.
Skip this cell if you just want the API server running.

In [ ]:
import urllib.request, zipfile, json, os, gc, sys, io, tempfile
from pathlib import Path
from datetime import datetime
import numpy as np
from PIL import Image

ZENODO_URL = "https://zenodo.org/records/5267549/files/test.zip?download=1"
DATA_DIR = Path("/kaggle/working/zenodo_eval")
DATA_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DATA_DIR / "test.zip"
def _download_with_retry(url, path, max_retries=3):
    """Download a file with retry and ZIP validation. Returns True on success."""
    import time
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading (attempt {attempt}/{max_retries})...")
            urllib.request.urlretrieve(url, str(path))
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"Downloaded {size_mb:.0f} MB")
            # Verify magic bytes (PK\x03\x04)
            with open(path, "rb") as f:
                magic = f.read(4)
            if magic != b"PK\x03\x04":
                print(f"Bad file: magic={magic.hex()} (expected PK\\x03\\x04), retrying...")
                path.unlink(missing_ok=True)
                continue
            # Try actually opening it
            try:
                with zipfile.ZipFile(str(path), "r") as _test:
                    _test.namelist()
                print("ZIP verification OK")
                return True
            except zipfile.BadZipFile as e:
                print(f"BadZipFile: {e}, retrying...")
                path.unlink(missing_ok=True)
                continue
        except Exception as e:
            print(f"Download error: {e}, retrying in 10s...")
            time.sleep(10)
            path.unlink(missing_ok=True)
    return False


if not zip_path.exists() or zip_path.stat().st_size == 0:
    ok = _download_with_retry(ZENODO_URL, zip_path)
    if not ok:
        print("ERROR: Failed to download valid test.zip after retries. Skipping evaluation.")
        garments = []
else:
    print("test.zip already exists")
    # Verify existing file
    try:
        with zipfile.ZipFile(str(zip_path), "r") as test:
            test.namelist()
        print("Existing ZIP valid")
    except zipfile.BadZipFile:
        print("Existing test.zip is corrupt, re-downloading...")
        zip_path.unlink(missing_ok=True)
        ok = _download_with_retry(ZENODO_URL, zip_path)
        if not ok:
            print("ERROR: Failed to download valid test.zip. Skipping evaluation.")
            garments = []
# Extract and index
extract_dir = DATA_DIR / "test"
extract_dir.mkdir(parents=True, exist_ok=True)
garments = []
with zipfile.ZipFile(str(zip_path), "r") as zf:
    names = zf.namelist()
    folders = set()
    for name in names:
        parts = name.strip("/").split("/")
        if len(parts) >= 2:
            folders.add("/".join(parts[:2]))
    print(f"Extracting {len(folders)} garments...")
    zf.extractall(str(extract_dir))

for folder in sorted(folders):
    fp = extract_dir / folder
    if not fp.is_dir(): continue
    objs = sorted(fp.glob("*.obj"))
    if objs:
        garments.append({"id": folder.replace("/","_"), "folder": str(fp), "obj": str(objs[0])})

print(f"Found {len(garments)} garments")

# Limit for quick test
LIMIT = 10
garments = garments[:LIMIT]
print(f"Evaluating {len(garments)}...")

# Render + Chamfer
import trimesh
from scipy.spatial import KDTree

results = []
for i, g in enumerate(garments):
    print(f"  [{i+1}/{len(garments)}] {g['id']}...")
    r = {"id": g["id"], "errors": [], "metrics": {}}
    try:
        gt = trimesh.load(g["obj"])
        gv = gt.vertices
        r["metrics"]["gt_verts"] = len(gv)
        # Simple reconstruction sim: just report GT stats for now
        # Full GarmentRec eval needs the model loaded from Cell 4
        r["metrics"]["gt_faces"] = len(gt.faces)
    except Exception as e:
        r["errors"].append(str(e)[:100])
    results.append(r)

print("\n=== RESULTS ===")
for r in results:
    print(f"  {r['id']}: {r['metrics']}")

out = DATA_DIR / "eval_results.json"
with open(out, "w") as f:
    json.dump({"timestamp": str(datetime.utcnow()), "n": len(results), "results": results}, f, indent=2)
print(f"\nSaved to {out}")
